# 02 Predicting active and inactive users

For the week after each snapshot, this notebook predicts whether the user returns and what share of the cards shown the user clicks. It combines the two into the activity score and NetEase's inactive label, and measures how much history a prediction needs.

Users are split 70% training, 15% validation and 15% test. Training and validation use days 1 to 15, and the test set days 16 to 23. Input: `parquet/panel.parquet`. Outputs: `10_*`, `13_*`, `18_*`, `19_*`, `26_*`.

The slow steps are functions in `src/thesis/heavy/prediction.py`. `ensure(step)` reads a step's saved results from `outputs/` and runs the step only when a result is missing.

## 1. Setup

In [1]:
# Libraries and shared code
import numpy as np
import pandas as pd
from IPython.display import display
from thesis.cache import ensure
from thesis.panel import feature_names
from thesis.heavy.prediction import feature_families, feature_groups, LABEL_METRICS as METRICS, LABEL_CMETRICS as CMETRICS

In [2]:
# The panel, its splits and the 59 inputs (counts as in the saved model results)
R10 = ensure("02.models")["10_panel_models"]
FEATS = feature_names()
pd.Series({**{f"rows_{k}": v for k, v in R10["rows"].items()}, "features": len(FEATS)})

[02.models] saved results loaded


rows_train    1952597
rows_valid     415419
rows_test      228295
features           59
dtype: int64

In [3]:
# How strongly the inputs move together: Spearman correlations between all pairs of the 59 inputs on training snapshots,
# and a few pairs that the scorecard reads together (counts of the same behavior over different spans)
# (thesis.heavy.prediction.correlations)
corr = ensure("02.correlations")["10_feature_correlations"]
pd.Series(corr["selected_pairs"], name="spearman").to_frame()

[02.correlations] saved results loaded


,spearman
w7_sessions ~ h_sessions,0.944
w7_visit_days ~ w7_sessions,0.947
w7_impr ~ w7_sessions,0.839
h_impr ~ h_sessions,0.861
h_clicks ~ h_likes,0.385
w7_visit_days ~ h_visit_days,0.933


## 2. Models

A constant, a one-feature rule, a scorecard on ten counts, a logistic regression on all 59 inputs and LightGBM. Settings are chosen on validation users by the log-loss.

In [4]:
# LightGBM tree size and logistic regression penalty chosen on validation users (thesis.heavy.prediction.tuning)
TUNE = ensure("02.tuning")["10_tuning"]
pd.concat({(t, m): pd.DataFrame(TUNE[t][m]["grid"]) for t in ("R", "C") for m in ("lightgbm", "logistic")})

[02.tuning] saved results loaded


num_leaves  min_data_in_leaf   trees  valid_logloss  seconds  \
R lightgbm 0          7.0              50.0  2778.0       0.533446     72.1   
           1          7.0             200.0  2646.0       0.533454     71.2   
           2          7.0            1000.0  3000.0       0.533293     81.6   
           3          7.0            5000.0  2991.0       0.533264     86.5   
           4         15.0              50.0  1868.0       0.533313     59.0   
           5         15.0             200.0  1868.0       0.533196     59.8   
           6         15.0            1000.0  1681.0       0.533126     56.7   
           7         15.0            5000.0  1796.0       0.533101     65.8   
           8         63.0              50.0   446.0       0.533210     26.8   
           9         63.0             200.0   698.0       0.533015     38.8   
           10        63.0            1000.0   593.0       0.533010     38.0   
           11        63.0            5000.0   629.0       0.532989     52.0   
           12       255.0              50.0   161.0       0.533107     26.1   
           13       255.0             200.0   208.0       0.533013     30.9   
           14       255.0            1000.0   186.0       0.533032     31.8   
           15       255.0            5000.0   323.0       0.533174     56.9   
  logistic 0          NaN               NaN     NaN       0.543483      NaN   
           1          NaN               NaN     NaN       0.543486      NaN   
           2          NaN               NaN     NaN       0.543489      NaN   
           3          NaN               NaN     NaN       0.543488      NaN   
C lightgbm 0          7.0              50.0   694.0       0.182551     12.1   
           1          7.0             200.0   647.0       0.182557     11.7   
           2          7.0            1000.0   936.0       0.182526     16.6   
           3          7.0            5000.0   887.0       0.182529     16.8   
           4         15.0              50.0   495.0       0.182511     11.1   
           5         15.0             200.0   626.0       0.182505     13.7   
           6         15.0            1000.0   312.0       0.182531      8.4   
           7         15.0            5000.0   540.0       0.182490     14.6   
           8         63.0              50.0   231.0       0.182529     10.4   
           9         63.0             200.0   216.0       0.182513     10.5   
           10        63.0            1000.0   124.0       0.182519      8.2   
           11        63.0            5000.0   115.0       0.182504      8.9   
           12       255.0              50.0    89.0       0.182586     11.7   
           13       255.0             200.0    97.0       0.182567     12.8   
           14       255.0            1000.0    81.0       0.182563     13.1   
           15       255.0            5000.0    83.0       0.182522     10.4   
  logistic 0          NaN               NaN     NaN       0.183079      NaN   
           1          NaN               NaN     NaN       0.183080      NaN   
           2          NaN               NaN     NaN       0.183080      NaN   
           3          NaN               NaN     NaN       0.183079      NaN   

                   C  
R lightgbm 0     NaN  
           1     NaN  
           2     NaN  
           3     NaN  
           4     NaN  
           5     NaN  
           6     NaN  
           7     NaN  
           8     NaN  
           9     NaN  
           10    NaN  
           11    NaN  
           12    NaN  
           13    NaN  
           14    NaN  
           15    NaN  
  logistic 0    0.01  
           1    0.10  
           2    1.00  
           3   10.00  
C lightgbm 0     NaN  
           1     NaN  
           2     NaN  
           3     NaN  
           4     NaN  
           5     NaN  
           6     NaN  
           7     NaN  
           8     NaN  
           9     NaN  
           10    NaN  
           11    NaN  
           12   

In [5]:
# Chosen settings
pd.DataFrame({t: {**TUNE[t]["lightgbm"]["chosen"], "lightgbm_at_grid_edge": TUNE[t]["lightgbm"]["at_grid_edge"],
                  "logistic_C": TUNE[t]["logistic"]["chosen_C"], "logistic_at_grid_edge": TUNE[t]["logistic"]["at_grid_edge"]}
              for t in ("R", "C")})

,R,C
num_leaves,63,15
min_data_in_leaf,5000,5000
lightgbm_at_grid_edge,[min_data_in_leaf],[min_data_in_leaf]
logistic_C,0.01,0.01
logistic_at_grid_edge,True,True


In [6]:
# Test scores of every model for both outcomes (thesis.heavy.prediction.models; predictions in 10_predictions)
pd.concat({t: pd.DataFrame({m: v for m, v in R10[t].items() if m in ("constant", "rule", "scorecard", "logistic", "lightgbm")}).T
           for t in ("R_return", "C_click_propensity")})

pseudo_r2     auc  lift_no_return_bottom10
R_return           constant      0.0000  0.5000                      NaN
                   rule          0.2021  0.7702                      NaN
                   scorecard     0.2362  0.8149                      NaN
                   logistic      0.2524  0.8169                      NaN
                   lightgbm      0.2605  0.8191                    1.793
C_click_propensity constant      0.0000  0.5000                      NaN
                   rule          0.0764  0.7994                      NaN
                   scorecard     0.0880  0.8024                      NaN
                   logistic      0.0896  0.8035                      NaN
                   lightgbm      0.0916  0.8121                      NaN

In [7]:
# Marketing benchmark for return: the BG/NBD model on visit days alone, LightGBM given the same two inputs,
# and the main LightGBM, on the same test snapshots (thesis.heavy.prediction.bgnbd)
BG = ensure("02.bgnbd")["14_bgnbd"]
print("Chance of dropping out for good:", BG["dropout_probability"])
pd.DataFrame(BG["models"]).T

[02.models] saved results loaded


[02.bgnbd] computing


[02.bgnbd] saved in 131 s


Chance of dropping out for good: 5.774e-11


,auc_return,auc_ci95,auc_minus_lightgbm_ci95
bgnbd,0.7938,"[0.7913, 0.7961]","[-0.0264, -0.0241]"
nbd_no_dropout,0.7938,"[0.7913, 0.7961]","[-0.0264, -0.0241]"
lightgbm_frequency_recency_only,0.8013,"[0.7988, 0.8034]","[-0.0187, -0.017]"
lightgbm,0.8191,"[0.8169, 0.8212]","[0.0, 0.0]"


In [8]:
# Scorecard coefficients (per standard deviation of each input, log-odds) and accuracy by days since first seen
display(pd.DataFrame({t: R10[t]["scorecard_coefficients_per_sd"] for t in ("R_return", "C_click_propensity")}))
pd.concat({t: pd.DataFrame(R10[t]["by_history_days"]).T for t in ("R_return", "C_click_propensity")})

,R_return,C_click_propensity
w7_visit_days,-0.2383,-0.0206
h_visit_days,0.2638,-0.0057
w7_sessions,0.5448,-0.0241
h_sessions,0.8688,-0.0160
gap_since_prev_visit,0.0004,0.0006
history_days,-0.4099,0.0098
w7_impr,-0.0044,0.0271
h_impr,0.5904,0.0035
h_ctr,-0.0076,-0.1570
h_mean_pos,-0.3014,0.0227


rows  lightgbm_pseudo_r2  scorecard_pseudo_r2  \
R_return           1      63602.0              0.0559               0.0220   
                   2-3     8506.0              0.1456               0.0684   
                   4-7    14878.0              0.1938               0.1546   
                   8-14   39223.0              0.2399               0.2216   
                   15+   102086.0              0.3078               0.2918   
C_click_propensity 1      16189.0              0.0526               0.0440   
                   2-3     4143.0              0.0897               0.0881   
                   4-7     7726.0              0.0609               0.0578   
                   8-14   22437.0              0.0834               0.0800   
                   15+    78706.0              0.0969               0.0935   

                         rule_pseudo_r2  lightgbm_auc_return  rule_auc_return  
R_return           1            -0.0104               0.5538           0.5000  
                   2-3           0.0601               0.6174           0.5400  
                   4-7           0.0625               0.6701           0.6428  
                   8-14          0.2160               0.7253           0.6879  
                   15+           0.2442               0.8431           0.8018  
C_click_propensity 1             0.0183                  NaN              NaN  
                   2-3           0.0473                  NaN              NaN  
                   4-7           0.0281                  NaN              NaN  
                   8-14          0.0604                  NaN              NaN  
                   15+           0.0853                  NaN              NaN

## 3. Calibration

Predicted against observed rates overall, by tenth of predicted probability, by test day and by length of history.

In [9]:
# Calibration: predicted minus observed rate, intercept and slope, with bootstrap intervals over users
# (thesis.heavy.prediction.calibration)
R13 = ensure("02.calibration")["13_calibration"]
MODELS = ["scorecard", "logistic", "lightgbm"]
pd.concat({(tgt, m): pd.Series({k: v for k, v in R13[tgt][m].items() if not isinstance(v, (dict, list))})
           for tgt in ("R", "C") for m in MODELS}, axis=1)

[02.calibration] saved results loaded


R                           C                  
                  scorecard logistic lightgbm scorecard logistic lightgbm
predicted_rate       0.6430   0.6151   0.6014    0.0519   0.0526   0.0525
pred_minus_obs_pp    7.7054   4.9170   3.5502   -0.1656  -0.1037  -0.1122
intercept           -0.4618  -0.2948  -0.2122    0.0345   0.0215   0.0233
slope                0.9486   0.9508   0.9576    1.0126   1.0148   0.9880

In [10]:
# Recalibration on the latest week: one log-odds shift per return model, fitted on validation users' snapshots from
# days 9 to 15 and applied to the test period; the ranking (AUC) is unchanged (thesis.heavy.prediction.recalibration)
REC = ensure("02.recalibration")["13_recalibration"]
pd.DataFrame({m: {k: v for k, v in REC[m].items() if not isinstance(v, (dict, bool))} for m in MODELS})

[02.recalibration] saved results loaded


,scorecard,logistic,lightgbm
logodds_shift,0.0944,0.1613,0.0346
pred_minus_obs_pp_before,7.705,4.917,3.55
pred_minus_obs_pp_after,9.272,7.607,4.131
pred_minus_obs_pp_after_ci95,"[9.085, 9.463]","[7.422, 7.791]","[3.953, 4.304]"
pseudo_r2_before,0.2362,0.2524,0.2605
pseudo_r2_after,0.2245,0.2377,0.2586
pseudo_r2_gain_ci95,"[-0.012, -0.0115]","[-0.0152, -0.0143]","[-0.002, -0.0019]"


In [11]:
# Overestimate of return by days since first seen, validation and test snapshots (percentage points)
ensure("02.calibration")["13_calibration_by_history"]

[02.calibration] saved results loaded


,set,history_days,snapshots,observed_return_pct,lightgbm_pred_minus_obs_pp,scorecard_pred_minus_obs_pp
0,test,1,63602,25.45,5.29,11.94
1,test,2-3,8506,48.71,6.55,17.49
2,test,4-7,14878,51.93,5.94,12.69
3,test,8-15,50814,59.21,3.93,6.82
4,test,16-19,54505,76.76,1.40,3.49
5,test,20-23,35990,81.18,1.50,3.48
6,valid,1,204461,38.46,0.24,0.24
7,valid,2-3,46285,70.33,-0.23,-0.17
8,valid,4-7,71134,73.40,0.25,0.32
9,valid,8-15,93539,78.94,-0.13,-0.15


## 4. Early actions and stability over time

LightGBM refitted on groups of features, also by days since first seen, a model trained only on days 1 to 8, and a reference for how predictable clicks are.

In [12]:
# Feature groups by name (thesis.heavy.prediction.family): visits and exposure = the activity, exposure and calendar
# groups of thesis.panel.FEATURE_GROUPS
G = feature_families(FEATS)
GROUPS = feature_groups(FEATS)
pd.Series({g: ", ".join(f) for g, f in G.items()}, name="features").to_frame()

,features
visits,"history_days, gap_since_prev_visit, w1_impr, w..."
clicks,"w1_clicks, w1_viewtime, w3_clicks, w3_viewtime..."
social,"w1_likes, w1_shares, w1_viewcomments, w3_likes..."


In [13]:
# LightGBM refitted on each group and on the out-of-time samples, every subgroup scored with bootstrap intervals, and
# the same-week ceiling for clicks (thesis.heavy.prediction.robustness)
R18 = ensure("02.robustness")["18_rq2_robustness"]
MODELS18 = ["all_features_step10", "logistic_step10", "scorecard_step10", "baseline", *GROUPS,
            "strict_out_of_time_days1_8", "same_size_control_days1_15"]


def table(sub):
    out = {}
    for t in ("R_return", "C_click_propensity"):
        e = R18[t]["subsets"][sub]
        for m in MODELS18:
            for met in ("pseudo_r2", "auc"):
                if f"{m}_{met}" in e:
                    out.setdefault(m, {})[(t, met)] = e[f"{m}_{met}"]
    return pd.DataFrame(out).T


# Accuracy by feature group and out-of-time fit, all test rows
table("all")

[02.robustness] saved results loaded


R_return         C_click_propensity        
                           pseudo_r2     auc          pseudo_r2     auc
all_features_step10           0.2605  0.8191             0.0916  0.8121
logistic_step10               0.2524  0.8169             0.0896  0.8035
scorecard_step10              0.2362  0.8149             0.0880  0.8024
baseline                      0.2021  0.7702             0.0764  0.7994
visits_exposure               0.2601  0.8193             0.0667  0.7987
visits_exposure+clicks        0.2606  0.8194             0.0914  0.8123
clicks_social_only            0.2026  0.8066             0.0890  0.8005
strict_out_of_time_days1_8    0.2498  0.8143             0.0905  0.8109
same_size_control_days1_15    0.2605  0.8185             0.0911  0.8123

In [14]:
# Early actions by days since first seen: AUC of each feature group minus the full model (95% intervals)
pd.concat({t: pd.DataFrame({s: {g: R18[t]["subsets"][s].get(f"{g}_auc_minus_all_ci95") for g in GROUPS}
                            for s in R18[t]["subsets"]}).T for t in ("R_return", "C_click_propensity")})

visits_exposure visits_exposure+clicks  \
R_return           all               [-0.0, 0.0003]       [0.0002, 0.0004]   
                   history_1       [-0.001, 0.0009]      [-0.0002, 0.0012]   
                   history_2-3     [-0.001, 0.0046]      [-0.0013, 0.0025]   
                   history_4-7    [-0.0013, 0.0014]      [-0.0016, 0.0004]   
                   history_8-14   [-0.0001, 0.0009]      [-0.0002, 0.0007]   
                   history_15+      [-0.0005, -0.0]       [0.0001, 0.0005]   
                   new_accounts   [-0.0015, 0.0011]      [-0.0008, 0.0014]   
C_click_propensity all             [-0.0168, -0.01]      [-0.0002, 0.0004]   
                   history_1     [-0.0425, -0.0185]      [-0.0051, 0.0002]   
                   history_2-3   [-0.0377, -0.0006]      [-0.0046, 0.0028]   
                   history_4-7   [-0.0446, -0.0115]      [-0.0033, 0.0008]   
                   history_8-14  [-0.0332, -0.0144]      [-0.0006, 0.0015]   
                   history_15+   [-0.0148, -0.0073]      [-0.0002, 0.0005]   
                   new_accounts    [-0.0353, 0.005]      [-0.0036, 0.0024]   

                                 clicks_social_only  
R_return           all           [-0.0134, -0.0118]  
                   history_1     [-0.0248, -0.0131]  
                   history_2-3   [-0.0811, -0.0502]  
                   history_4-7   [-0.0489, -0.0338]  
                   history_8-14  [-0.0339, -0.0277]  
                   history_15+   [-0.0276, -0.0243]  
                   new_accounts  [-0.0322, -0.0199]  
C_click_propensity all           [-0.0139, -0.0093]  
                   history_1     [-0.0558, -0.0183]  
                   history_2-3    [-0.0366, 0.0075]  
                   history_4-7   [-0.0414, -0.0091]  
                   history_8-14    [-0.0246, -0.01]  
                   history_15+     [-0.0152, -0.01]  
                   new_accounts  [-0.0329, -0.0038]

In [15]:
# Ceiling for click propensity
pd.Series(R18["D_predictability_ceiling_clicks"], name="value").to_frame()

,value
origins_with_both_halves,125970.0000
ceiling_same_week_half_split_pseudo_r2,0.1205
step10_model_on_the_same_half_pseudo_r2,0.0919


## 5. The inactive label

The activity score is the chance of returning times the chance of clicking a card shown. A snapshot is labeled inactive when its score is in the lowest 10% of its day. The same label is also built from the scorecard.

In [16]:
# Activity score for every test origin from LightGBM and from the scorecard; next week for labeled and other snapshots
# at cutoffs from 5% to 50%; the label as a yes/no prediction (thesis.heavy.prediction.label; scores in 19_scores)
L19 = ensure("02.label")
res = L19["19_inactive_label"]
display(pd.DataFrame(res["ranking"]).T)
pd.concat({q: pd.DataFrame({g: {k: v[g][k] for k in METRICS} for g in v}).T for q, v in res["by_threshold"].items()})

[02.label] saved results loaded


,base_rate,auc,auc_ci95,auc_scorecard,auc_scorecard_ci95
zero_clicks,0.8552,0.7791,"[0.7736, 0.7853]",0.7643,"[0.759, 0.7699]"
no_return,0.4341,0.6107,"[0.6069, 0.6144]",0.6303,"[0.6273, 0.6336]"


share  return_rate  mean_clicks  zero_clicks_share  \
bottom_5pct  inactive   0.05       0.6715       0.1723             0.9200   
             other      0.95       0.5604       1.2698             0.8518   
bottom_10pct inactive   0.10       0.5637       0.1427             0.9352   
             other      0.90       0.5662       1.3340             0.8463   
bottom_20pct inactive   0.20       0.5005       0.1317             0.9441   
             other      0.80       0.5823       1.4857             0.8330   
bottom_25pct inactive   0.25       0.4878       0.1290             0.9452   
             other      0.75       0.5920       1.5769             0.8252   
bottom_30pct inactive   0.30       0.4798       0.1301             0.9462   
             other      0.70       0.6028       1.6798             0.8162   
bottom_40pct inactive   0.40       0.4775       0.1321             0.9459   
             other      0.60       0.6249       1.9368             0.7947   
bottom_50pct inactive   0.50       0.4810       0.1390             0.9447   
             other      0.50       0.6509       2.2908             0.7657   

                       click_rate_if_returned  
bottom_5pct  inactive                  0.0047  
             other                     0.0579  
bottom_10pct inactive                  0.0059  
             other                     0.0593  
bottom_20pct inactive                  0.0081  
             other                     0.0613  
bottom_25pct inactive                  0.0089  
             other                     0.0622  
bottom_30pct inactive                  0.0097  
             other                     0.0630  
bottom_40pct inactive                  0.0110  
             other                     0.0650  
bottom_50pct inactive                  0.0124  
             other                     0.0672

In [17]:
# What fills each label: the spread of the two predictions on the log scale, and whether a labeled snapshot is also in
# the bottom 10% of its day by the return prediction alone, by the click prediction alone, by both or by neither
comp = L19["19_label_composition"]
pd.DataFrame({k: v for k, v in comp.items() if isinstance(v, dict)})

,lightgbm,scorecard
sd_log_p_return,0.49400,0.41700
sd_log_p_click,0.67600,0.55500
labelled_low_return_only,0.37580,0.30090
labelled_low_click_only,0.43950,0.14500
labelled_low_both,0.00000,0.00010
labelled_low_neither,0.18470,0.55400
labelled_mean_p_return,0.57210,0.44640
labelled_mean_p_click,0.01723,0.02437
low_click_mean_p_return,0.95540,0.95920
low_return_mean_p_click,0.03829,0.04031


In [18]:
# The label as a yes/no prediction at each cutoff, and the scorecard label at 10%
display(pd.concat({q: pd.DataFrame({t: {k: v[t][k] for k in CMETRICS} for t in v}).T
                   for q, v in res["classification_by_threshold"].items()}))
pd.DataFrame({k: v for k, v in res["scorecard_label_bottom10"].items() if isinstance(v, dict)}).T

precision  recall    lift
bottom_5pct  zero_clicks     0.9200  0.0538  1.0758
             no_return       0.3285  0.0378  0.7567
bottom_10pct zero_clicks     0.9352  0.1093  1.0936
             no_return       0.4363  0.1005  1.0051
bottom_20pct zero_clicks     0.9441  0.2208  1.1039
             no_return       0.4995  0.2301  1.1508
bottom_25pct zero_clicks     0.9452  0.2763  1.1052
             no_return       0.5122  0.2950  1.1801
bottom_30pct zero_clicks     0.9462  0.3319  1.1064
             no_return       0.5202  0.3595  1.1983
bottom_40pct zero_clicks     0.9459  0.4424  1.1060
             no_return       0.5225  0.4815  1.2037
bottom_50pct zero_clicks     0.9447  0.5523  1.1047
             no_return       0.5190  0.5979  1.1958

,share,return_rate,mean_clicks,zero_clicks_share,click_rate_if_returned
inactive,0.1,0.3902,0.1382,0.9511,0.0114
other,0.9,0.5855,1.3345,0.8446,0.0560


## 6. The cutoff

Next week's outcomes by percentile band of the score, and a search for a natural break.

In [19]:
# Next-week outcomes by percentile band of the score, and the break-point search
by_band = L19["19_score_bands"].set_index("band")
cut_res = L19["19_cutoff_choice"]
pd.Series({k: str(v) for k, v in cut_res.items()}, name="value").to_frame()

,value
return_rate_break_point_by_fit_range,"{'bands_0_39': 5, 'bands_0_59': 5, 'bands_0_79..."
last_band_before_zero_click_share_first_drops_below_95pct,-1
bands_0_to_24,"{'return_rate_range': [0.4199, 0.9772], 'mean_..."


In [20]:
# Bands 0-29 and every fifth band after
pd.concat([by_band.iloc[:30], by_band.iloc[30::5]]).round(4)

,user_days,return_rate,mean_clicks,zero_clicks_share,mean_score
band,,,,,
0,2280,0.9772,0.2662,0.8768,0.0036
1,2283,0.7714,0.1735,0.9137,0.0047
2,2281,0.6116,0.1670,0.9259,0.0053
3,2283,0.5094,0.1388,0.9387,0.0058
4,2284,0.4886,0.1160,0.9448,0.0061
5,2283,0.4731,0.1178,0.9474,0.0064
6,2283,0.4529,0.1016,0.9536,0.0066
7,2283,0.4639,0.1213,0.9531,0.0069
8,2282,0.4500,0.1030,0.9514,0.0071


## 7. Learning curve

The same users, first seen on days 2 to 9, are scored after 1 to 15 days with one model per day. A second group, first seen on days 16 and 17, is scored by models trained on days 1 to 15 only. At 1, 3 and 7 days the return model is also refitted on groups of features.

In [21]:
# The same users scored after 1 to 15 days with one model per day (LightGBM, scorecard, logistic regression, baseline),
# accuracy at each horizon with bootstrap intervals and the gain from each extra day (thesis.heavy.prediction.history)
H = ensure("02.history")
CURVES = {g: {t: {int(d): e for d, e in H["26_history_curve.json"][g][t].items()} for t in ("R_return", "C_click_propensity")}
          for g in ("user_split", "strict_time_split")}
pd.concat({(g, t): pd.DataFrame({d: {k: v for k, v in e.items() if "gain" not in k and not isinstance(v, list)}
                                 for d, e in CURVES[g][t].items()}).T
           for g in CURVES for t in ("R_return", "C_click_propensity")})

[02.history] saved results loaded


trees  return_next7  \
user_split        R_return           1   305.0        0.3858   
                                     2   245.0        0.3706   
                                     3   324.0        0.3625   
                                     4   275.0        0.3558   
                                     5   280.0        0.3509   
                                     6   190.0        0.3472   
                                     7   195.0        0.3455   
                                     8   218.0        0.3439   
                                     9   197.0        0.3416   
                                     10  252.0        0.3397   
                                     11  275.0        0.3374   
                                     12  173.0        0.3340   
                                     13  198.0        0.3321   
                                     14  163.0        0.3295   
                                     15  173.0        0.3266   
                  C_click_propensity 1   243.0        0.3858   
                                     2   127.0        0.3706   
                                     3   171.0        0.3625   
                                     4   135.0        0.3558   
                                     5   167.0        0.3509   
                                     6   135.0        0.3472   
                                     7   120.0        0.3455   
                                     8   179.0        0.3439   
                                     9   134.0        0.3416   
                                     10  221.0        0.3397   
                                     11  168.0        0.3374   
                                     12  291.0        0.3340   
                                     13  138.0        0.3321   
                                     14   95.0        0.3295   
                                     15  136.0        0.3266   
strict_time_split R_return           1   305.0        0.2553   
                                     2   245.0        0.2380   
                                     3   324.0        0.2315   
                                     4   275.0        0.2288   
                                     5   280.0        0.2248   
                                     6   190.0        0.2213   
                                     7   195.0        0.2209   
                  C_click_propensity 1   243.0        0.2553   
                                     2   127.0        0.2380   
                                     3   171.0        0.2315   
                                     4   135.0        0.2288   
                                     5   167.0        0.2248   
                                     6   135.0        0.2213   
                                     7   120.0        0.2209   

                                         users_20plus_cards_next7  \
user_split        R_return           1                    10386.0   
                                     2                     9931.0   
                                     3                     9705.0   
                                     4                     9639.0   
                                     5                     9620.0   
                                     6                     9491.0   
                                     7                     9453.0   
                                     8                     9358.0   
                                     9                     9322.0   
                                     10                    9065.0   
                                     11                    8865.0   
                                     12                    8647.0   
                                     13                    8371.0   
                                     14                    8034.0   
                                     15                    7618.0   
                  C_click_propensity 1                  

In [22]:
# The gain in return AUC and click AUC from one more observed day (95% intervals)
pd.concat({g: pd.DataFrame({d: {"return_auc_gain": CURVES[g]["R_return"][d]["lightgbm_auc_gain_from_previous_day_ci95"],
                                "click_auc_gain": CURVES[g]["C_click_propensity"][d]["lightgbm_auc_gain_from_previous_day_ci95"]}
                            for d in CURVES[g]["R_return"] if d > 1}).T for g in CURVES})

return_auc_gain     click_auc_gain
user_split        2   [0.0248, 0.0294]   [0.0114, 0.0284]
                  3   [0.0138, 0.0173]   [0.0052, 0.0202]
                  4   [0.0099, 0.0136]   [0.0043, 0.0184]
                  5   [0.0071, 0.0108]    [0.004, 0.0174]
                  6   [0.0069, 0.0104]  [-0.0037, 0.0105]
                  7    [0.006, 0.0097]    [0.0041, 0.017]
                  8    [0.0052, 0.009]  [-0.0057, 0.0082]
                  9   [0.0025, 0.0058]   [-0.003, 0.0116]
                  10  [0.0043, 0.0076]   [-0.001, 0.0118]
                  11  [0.0026, 0.0058]  [-0.0027, 0.0097]
                  12  [0.0014, 0.0045]  [-0.0027, 0.0097]
                  13  [0.0034, 0.0065]  [-0.0021, 0.0098]
                  14  [0.0028, 0.0061]     [-0.0, 0.0133]
                  15  [0.0027, 0.0062]  [-0.0085, 0.0071]
strict_time_split 2   [0.0224, 0.0364]   [-0.0248, 0.032]
                  3   [0.0064, 0.0189]  [-0.0306, 0.0304]
                  4    [0.006, 0.0176]  [-0.0169, 0.0539]
                  5    [0.0014, 0.012]   [0.0004, 0.0725]
                  6     [-0.0, 0.0108]    [-0.044, 0.025]
                  7   [0.0013, 0.0149]  [-0.0393, 0.0265]

In [23]:
# Early actions among users in their first days: return model refitted on feature groups at 1, 3 and 7 days
early = H["26_early_actions"]
pd.concat({d: pd.DataFrame(v).T for d, v in early.items()})

auc pseudo_r2  auc_minus_all_ci95  \
1 all                     0.6383    0.0469                 NaN   
  visits_exposure         0.6383    0.0468   [-0.0005, 0.0004]   
  visits_exposure_clicks  0.6383    0.0468   [-0.0003, 0.0003]   
3 all                     0.6811    0.0855                 NaN   
  visits_exposure         0.6814    0.0855   [-0.0001, 0.0007]   
  visits_exposure_clicks   0.681    0.0854   [-0.0004, 0.0002]   
7 all                     0.7182    0.1234                 NaN   
  visits_exposure         0.7179    0.1231  [-0.0006, -0.0001]   
  visits_exposure_clicks   0.718    0.1233      [-0.0004, 0.0]   

                         pseudo_r2_minus_all_ci95  
1 all                                         NaN  
  visits_exposure               [-0.0003, 0.0002]  
  visits_exposure_clicks        [-0.0002, 0.0001]  
3 all                                         NaN  
  visits_exposure               [-0.0002, 0.0003]  
  visits_exposure_clicks        [-0.0003, 0.0001]  
7 all                                         NaN  
  visits_exposure              [-0.0005, -0.0001]  
  visits_exposure_clicks        [-0.0003, 0.0001]

## 8. Summary

Return is predicted well and clicks modestly. A scorecard of a few counts comes close to LightGBM. Visits carry the signal for return, likes and shares add nothing, and prediction is weakest in a user's first days.